# Notebook 2 — Frame Collection & the G1 Adoption-Detection Validation Gate
### Study: *The Second-Order Compute Footprint of AI-Coding-Tool Adoption in Open Source*

Notebook 1 froze the plan and proved feasibility. This notebook does the two things that license the rest of the study:

1. **Collect a stratified sampling frame** (~2,000 repos: 3 popularity bands × 5 languages), resumable and checkpointed to Drive, with a **software-project filter** that removes the awesome-list / book / curriculum repos that contaminated the probe.
2. **Run the formal G1 validation gate** — the measurement-validity precondition from the pre-registration. We draw a stratified **gold set (~250 repos)**, export **blinded local labeling sheets** for two independent coders, run a **calibration round**, then compute **Cohen's κ** (inter-coder reliability) and the **detector precision / recall / F1 with Wilson 95% CIs**. The gate: high-confidence signals must reach **precision ≥ 0.90** to proceed to RQ1–RQ3.

**Design decisions locked from Notebook 1's probe (logged as deviations):**
- *Outcomes split by data source.* Activity outcomes (commits, churn, PRs) use git/PR history (does not age out) → full 2022–2025 window. The CI-minutes/carbon outcome inherits the empirically-observed **~14-month GitHub Actions retention ceiling** (probe: median 12.15, max 14.0 months) → scoped to recent adopters. This notebook records each repo's retrievable CI depth so Notebook 3 can apply that scoping.
- *Software-project filter* added to the frame (probe showed top-starred sort pulls in non-software repos that inflate false-positive adoption signals).
- *Stratified sampling* across popularity bands replaces top-starred-only (external-validity threat from the probe).

**Standards:** ACM SIGSOFT Empirical Standards — *Repository Mining* (sampling frame, replication), *Data Science*, and for G1 the human-coded-construct expectations (independent coders, codebook, κ).

> **Bug fixed from NB1:** the JSONL writer now emits real newlines (`json.dump` + `"\n"`), and a tolerant loader is used everywhere, so the delimiter problem cannot recur.


---
## Cell 0 — Re-attach Drive, config, and the resilient GitHub client

This re-establishes everything Notebook 1 set up (Drive paths, `CONFIG`, the cached rate-limited client). Run Notebook 1 first at least once so `config.json` and the pre-registration exist. The token self-test fails loudly here if auth is bad.


In [ ]:

%pip install -q requests requests-cache tenacity pandas pyarrow tqdm statsmodels scipy

import os, json, time, math, pathlib, datetime, random
import requests, requests_cache
import pandas as pd
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

from google.colab import drive, userdata
drive.mount('/content/drive')

PROJECT_ROOT = pathlib.Path('/content/drive/MyDrive/ai_coding_carbon')
DIRS = {k: PROJECT_ROOT / k for k in
        ['raw', 'cache', 'panels', 'cohort', 'results', 'figures', 'logs', 'probe', 'frame', 'gold']}
for d in [PROJECT_ROOT, *DIRS.values()]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = json.loads((PROJECT_ROOT / 'config.json').read_text())
random.seed(CONFIG['random_seed'])

# ---- token (strip + self-test) ----
GITHUB_TOKEN = (userdata.get('GITHUB_TOKEN') or '').strip()
assert GITHUB_TOKEN, 'No GITHUB_TOKEN in Colab Secrets.'
API = 'https://api.github.com'
HEADERS = {'Accept': 'application/vnd.github+json',
           'X-GitHub-Api-Version': '2022-11-28',
           'Authorization': f'Bearer {GITHUB_TOKEN}'}
_p = requests.get(f'{API}/user', headers=HEADERS, timeout=30)
assert _p.status_code == 200, f'Auth failed {_p.status_code}: {_p.json().get("message")}'
print('Authenticated as:', _p.json()['login'])

# ---- cached, rate-limit-aware client ----
session = requests_cache.CachedSession(
    cache_name=str(DIRS['cache'] / 'github_http_cache'),
    backend='sqlite', expire_after=None, allowable_codes=(200,), stale_if_error=True)

class RateLimited(Exception): pass

@retry(retry=retry_if_exception_type(RateLimited),
       wait=wait_exponential(multiplier=2, min=4, max=120), stop=stop_after_attempt(8))
def gh_get(url, params=None):
    r = session.get(url, headers=HEADERS, params=params, timeout=60)
    if r.status_code in (403, 429):
        rem = r.headers.get('X-RateLimit-Remaining')
        if rem == '0' or 'rate limit' in r.text.lower() or 'secondary' in r.text.lower():
            reset = int(r.headers.get('X-RateLimit-Reset', time.time()+60))
            sl = max(reset - time.time(), 5) + 2
            print(f'  [rate limit] sleeping {sl:.0f}s'); time.sleep(sl); raise RateLimited()
    r.raise_for_status(); return r

def gh_json(url, params=None): return gh_get(url, params=params).json()

# ---- robust JSONL helpers (the NB1 bug fixed for good) ----
def append_jsonl(path, rec):
    with open(path, 'a') as f:
        f.write(json.dumps(rec)); f.write(chr(10))      # REAL newline

def load_jsonl(path):
    recs = []
    if not pathlib.Path(path).exists(): return recs
    for line in open(path):
        line = line.strip()
        if not line: continue
        try: recs.append(json.loads(line))
        except json.JSONDecodeError:
            # tolerant fallback: split on any literal backslash-n from older files
            for part in line.split('\\n'):
                part = part.strip()
                if part:
                    try: recs.append(json.loads(part))
                    except json.JSONDecodeError: pass
    return recs

def load_ledger(path, default):
    return json.loads(pathlib.Path(path).read_text()) if pathlib.Path(path).exists() else default
def save_ledger(path, obj):
    pathlib.Path(path).write_text(json.dumps(obj, indent=2))

print('Environment re-attached. Frame target:', CONFIG.get('frame_target', 'see Cell 1'))


---
## Cell 1 — Stratified sampling-frame parameters

We replace top-starred-only with **stratified sampling across three popularity bands** per language, so the frame spans the popularity distribution rather than only the head. This directly addresses the external-validity threat the probe exposed.

The bands are defined on stars; within each band we sample by *recent activity* to avoid only-old or only-new bias. These parameters are written back into `config.json` (deviation logged).


In [ ]:

# Three popularity bands (stars). Tunable; chosen to span head/mid/long-tail of active OSS.
POP_BANDS = {
    "high":   {"stars_min": 1000, "stars_max": None},
    "mid":    {"stars_min": 200,  "stars_max": 999},
    "low":    {"stars_min": CONFIG['min_stars'], "stars_max": 199},   # >=50
}
FRAME = {
    "target_total": 2000,
    "per_band_per_language": 135,   # 135 * 3 bands * 5 langs ~= 2025 candidates pre-filter
    "languages": CONFIG['languages'],
    "pop_bands": POP_BANDS,
}
CONFIG['frame'] = FRAME
(PROJECT_ROOT / 'config.json').write_text(json.dumps(CONFIG, indent=2))

# deviation log entry
dev = PROJECT_ROOT / 'DEVIATIONS_LOG.txt'
stamp = datetime.datetime.utcnow().strftime('%Y-%m-%d')
entry = (f"{stamp}  Frame uses stratified sampling across 3 popularity bands x 5 langs "
         f"(replaces top-starred-only) to address external validity (probe finding). "
         f"Software-project filter added. CI/carbon outcome scoped to ~14-month Actions "
         f"retention window (probe: median 12.15, max 14.0 mo); activity outcomes keep full window.\n")
with open(dev, 'a') as f: f.write(entry)
print('Frame params saved. Deviation logged:\n', entry)
print(json.dumps(FRAME, indent=2))


---
## Cell 2 — Discover candidate repos per (language × band), resumable

The Search API is the bottleneck (30 req/min, 1000-result cap per query). We page each (language × band) query by stars and checkpoint every page to Drive, so a disconnect resumes losslessly. We over-collect slightly because the software-project filter (Cell 3) will drop some.


In [ ]:

frame_raw_path = DIRS['frame'] / 'candidates_raw.jsonl'
frame_ledger_path = DIRS['frame'] / 'frame_ledger.json'
ledger = load_ledger(frame_ledger_path, {"done_queries": [], "collected": 0})

def search_repos(lang, band_name, band, want):
    """Page the search API for one (lang, band); return list of repo dicts."""
    smin, smax = band['stars_min'], band['stars_max']
    star_q = f">={smin}" if smax is None else f"{smin}..{smax}"
    q = (f"language:{lang} stars:{star_q} archived:false fork:false "
         f"pushed:>={CONFIG['study_window'][0]}")
    out, page = [], 1
    while len(out) < want and page <= 10:   # 10*100 = 1000 result cap
        data = gh_json(f"{API}/search/repositories",
                       params={"q": q, "sort": "stars", "order": "desc",
                               "per_page": 100, "page": page})
        items = data.get("items", [])
        if not items: break
        for it in items:
            out.append({
                "full_name": it["full_name"], "language": lang, "band": band_name,
                "stars": it["stargazers_count"], "created_at": it["created_at"],
                "pushed_at": it["pushed_at"], "size_kb": it.get("size", 0),
                "open_issues": it.get("open_issues_count", 0),
                "description": (it.get("description") or "")[:300],
            })
        page += 1
        time.sleep(2.2)   # stay under 30 search req/min
    return out[:want]

want = FRAME['per_band_per_language']
for lang in FRAME['languages']:
    for band_name, band in POP_BANDS.items():
        key = f"{lang}|{band_name}"
        if key in ledger['done_queries']:
            continue
        repos = search_repos(lang, band_name, band, want)
        for r in repos:
            append_jsonl(frame_raw_path, r)
        ledger['done_queries'].append(key)
        ledger['collected'] += len(repos)
        save_ledger(frame_ledger_path, ledger)
        print(f"  {key:20s} -> {len(repos):3d} repos  (total {ledger['collected']})")

cand = pd.DataFrame(load_jsonl(frame_raw_path)).drop_duplicates("full_name")
print(f"\nUnique candidates collected: {len(cand)}")
print(cand.groupby(['language','band']).size().unstack(fill_value=0))


---
## Cell 3 — Software-project filter & per-repo enrichment (resumable)

For each candidate we fetch the file tree, recent commits, and Actions history, then apply the **software-project filter**. A repo is kept only if it looks like real software, not a list/book/doc repo. The filter rules (documented for the paper):

- **Has real CI** that runs on code (Actions workflows present), **and**
- **Source-file ratio**: a minimum share of recognized source files in the tree, **and**
- **Not an awesome/doc/curriculum repo**: name/description doesn't match the awesome-list/book/roadmap pattern *unless* it also has substantial source + CI.

We also record, per repo: detected adoption signals (for G1), and the **retrievable Actions-history depth** (to scope the carbon outcome later).


In [ ]:

import re

SRC_EXT = {'.py','.js','.ts','.tsx','.jsx','.go','.java','.rb','.rs','.c','.cc','.cpp',
           '.h','.hpp','.cs','.kt','.swift','.scala','.php','.m','.mm'}
DOC_PATTERN = re.compile(r'awesome|free-programming|developer-roadmap|book|curriculum|'
                         r'cheat-?sheet|interview|tutorial|guide|roadmap|notes|'
                         r'top-charts|hello-algo|design-primer', re.I)

def get_tree_paths(full_name):
    repo = gh_json(f"{API}/repos/{full_name}")
    branch = repo.get("default_branch", "main")
    try:
        tree = gh_json(f"{API}/repos/{full_name}/git/trees/{branch}", params={"recursive":"1"})
    except Exception:
        return repo, [], True   # treat as truncated/unavailable
    paths = [t["path"] for t in tree.get("tree", []) if t.get("type")=="blob"]
    return repo, paths, tree.get("truncated", False)

def source_ratio(paths):
    if not paths: return 0.0
    src = sum(1 for p in paths if os.path.splitext(p)[1].lower() in SRC_EXT)
    return src / len(paths)

def has_actions_workflows(paths):
    return any(p.startswith(".github/workflows/") and p.endswith((".yml",".yaml")) for p in paths)

def detect_signals(full_name, paths):
    arts = []
    lp = {p.lower() for p in paths}
    for art in CONFIG["adoption_signals"]["config_artifacts"]:
        a = art.lower()
        if a in lp or any(p == a or p.startswith(a + "/") for p in lp):
            arts.append(art)
    trailers = []
    try:
        commits = gh_json(f"{API}/repos/{full_name}/commits", params={"per_page":100})
        for c in commits:
            msg = (c.get("commit",{}).get("message","") or "").lower()
            for t in CONFIG["adoption_signals"]["commit_trailers"]:
                if t in msg:
                    trailers.append({"sha": c["sha"][:10], "signal": t,
                                     "date": c["commit"]["author"]["date"]}); break
    except Exception: pass
    return arts, trailers

def actions_depth(full_name):
    try:
        first = gh_json(f"{API}/repos/{full_name}/actions/runs", params={"per_page":1})
        total = first.get("total_count",0)
        if total == 0: return {"has_runs":False,"total_runs":0,"oldest":None,"newest":None}
        newest = first["workflow_runs"][0]["created_at"] if first.get("workflow_runs") else None
        last_pg = max((total-1)//100 + 1, 1)
        last = gh_json(f"{API}/repos/{full_name}/actions/runs",
                       params={"per_page":100,"page":last_pg})
        runs = last.get("workflow_runs",[])
        oldest = runs[-1]["created_at"] if runs else None
        return {"has_runs":True,"total_runs":total,"oldest":oldest,"newest":newest}
    except Exception as e:
        return {"has_runs":None,"error":str(e)}

enriched_path = DIRS['frame'] / 'enriched.jsonl'
enrich_ledger_path = DIRS['frame'] / 'enrich_ledger.json'
eled = load_ledger(enrich_ledger_path, {"done": []})
done = set(eled["done"])

cand = pd.DataFrame(load_jsonl(frame_raw_path)).drop_duplicates("full_name")
todo = [r for r in cand.to_dict("records") if r["full_name"] not in done]
print(f"To enrich: {len(todo)} (already done {len(done)})")

for i, r in enumerate(todo):
    fn = r["full_name"]
    try:
        repo, paths, truncated = get_tree_paths(fn)
        sr = source_ratio(paths)
        has_ci = has_actions_workflows(paths)
        arts, trailers = detect_signals(fn, paths)
        depth = actions_depth(fn)
        is_doc = bool(DOC_PATTERN.search(fn) or DOC_PATTERN.search(r.get("description","") or ""))
        # SOFTWARE-PROJECT FILTER
        keep = has_ci and (sr >= 0.10) and (not is_doc or sr >= 0.30)
        rec = {**r, "default_branch": repo.get("default_branch"),
               "n_paths": len(paths), "source_ratio": round(sr,3),
               "has_ci_workflow": has_ci, "is_doc_like": is_doc, "tree_truncated": truncated,
               "config_artifacts": arts, "commit_trailer_hits": trailers,
               "any_adoption_signal": bool(arts or trailers),
               "high_conf_signal": bool(arts or trailers),  # s1+s2 are the high-conf set
               "actions_depth": depth, "keep_in_frame": bool(keep),
               "enriched_at": datetime.datetime.utcnow().isoformat()+"Z"}
        append_jsonl(enriched_path, rec)
    except Exception as e:
        append_jsonl(enriched_path, {**r, "error": str(e), "keep_in_frame": False})
    done.add(fn); eled["done"] = sorted(done)
    if i % 25 == 0:
        save_ledger(enrich_ledger_path, eled)
        print(f"  [{i+1}/{len(todo)}] {fn}")
save_ledger(enrich_ledger_path, eled)
print("Enrichment complete.")


---
## Cell 4 — Assemble the filtered sampling frame & report attrition

We report how many candidates survive each filter step (a Repository-Mining standard expectation: explicit inclusion/exclusion accounting), and save the final frame.


In [ ]:

enr = pd.DataFrame(load_jsonl(enriched_path)).drop_duplicates("full_name")
n0 = len(enr)
step_ci   = enr["has_ci_workflow"].fillna(False)
step_src  = enr["source_ratio"].fillna(0) >= 0.10
step_keep = enr["keep_in_frame"].fillna(False)

attrition = {
    "candidates_enriched": int(n0),
    "with_ci_workflow": int(step_ci.sum()),
    "with_source_ratio>=0.10": int(step_src.sum()),
    "passed_software_filter (final frame)": int(step_keep.sum()),
    "dropped_doc_like": int((enr["is_doc_like"].fillna(False) & ~step_keep).sum()),
}
print(json.dumps(attrition, indent=2))

frame = enr[step_keep].copy()
frame.to_parquet(DIRS['frame'] / 'frame_final.parquet', index=False)
print(f"\nFinal frame: {len(frame)} repos")
print(frame.groupby(['language','band']).size().unstack(fill_value=0))
print("\nAdoption-signal prevalence in frame:")
print(f"  any signal:        {frame['any_adoption_signal'].mean()*100:.1f}%")
print(f"  config artifact:   {(frame['config_artifacts'].map(len)>0).mean()*100:.1f}%")
print(f"  commit trailer:    {(frame['commit_trailer_hits'].map(len)>0).mean()*100:.1f}%")

# retrievable CI depth distribution (to scope carbon outcome in NB3)
def months_back(ts):
    if not ts: return None
    d = pd.to_datetime(ts, utc=True)
    return round((pd.Timestamp.now(tz='UTC')-d).days/30.44,1)
frame["ci_oldest_months_back"] = frame["actions_depth"].map(
    lambda a: months_back(a.get("oldest")) if isinstance(a,dict) else None)
print("\nRetrievable CI-history depth (months back):")
print(frame["ci_oldest_months_back"].describe())
(DIRS['frame'] / 'attrition.json').write_text(json.dumps(attrition, indent=2))


---
## Cell 5 — Draw the stratified gold set (~250) for G1

The gold set must let us estimate **precision** (of repos the detector flags, how many are truly adopters) and **recall** (of true adopters, how many the detector catches). So we **oversample detector-positive repos** (to pin down precision tightly) while retaining a random slice of detector-negatives (to estimate recall and catch false negatives). Stratified by language × band × detector-verdict. This sampling design is recorded for the paper.


In [ ]:

frame = pd.read_parquet(DIRS['frame'] / 'frame_final.parquet')
GOLD_N = 250
rng = random.Random(CONFIG['random_seed'])

pos = frame[frame['high_conf_signal']].copy()
neg = frame[~frame['high_conf_signal']].copy()

# Allocate ~60% positives (precision focus), ~40% negatives (recall/FN catch)
n_pos = min(int(GOLD_N*0.6), len(pos))
n_neg = min(GOLD_N - n_pos, len(neg))

def strat_sample(df, n):
    if n <= 0 or len(df)==0: return df.iloc[0:0]
    # proportional across language x band
    grp = df.groupby(['language','band'], group_keys=False)
    frac = n/len(df)
    out = grp.apply(lambda g: g.sample(max(1, round(len(g)*frac)), random_state=CONFIG['random_seed'])
                    if len(g)>0 else g)
    return out.sample(min(n, len(out)), random_state=CONFIG['random_seed'])

gold = pd.concat([strat_sample(pos, n_pos), strat_sample(neg, n_neg)]).drop_duplicates('full_name')
gold = gold.sample(frac=1, random_state=CONFIG['random_seed']).reset_index(drop=True)  # shuffle (blind order)
gold['repo_id'] = ['G%04d'%i for i in range(len(gold))]

gold.to_parquet(DIRS['gold'] / 'gold_set.parquet', index=False)
print(f"Gold set: {len(gold)} repos  (positives ~{n_pos}, negatives ~{n_neg})")
print(gold.groupby(['band','high_conf_signal']).size())


---
## Cell 6 — Export blinded local labeling sheets, codebook & calibration batch

Two **identical, blinded** CSVs (one per coder) are written to Drive for download. They show only neutral evidence — repo URL, the config artifacts found, the matching commit-message snippets, a README excerpt — and **not** the detector's verdict or the other coder's labels. Each coder fills `label` and `reason`, saves, and returns the file.

A **codebook** (the shared decision rules) and a **calibration batch** (~25 rows to code first, compare, refine) are written alongside.


In [ ]:

import csv, textwrap

gold = pd.read_parquet(DIRS['gold'] / 'gold_set.parquet')

def readme_excerpt(full_name, branch):
    for name in ("README.md","README.rst","readme.md","README"):
        try:
            r = gh_get(f"https://raw.githubusercontent.com/{full_name}/{branch}/{name}")
            return r.text[:800].replace("\n"," ").replace("\r"," ")
        except Exception: continue
    return ""

# Build evidence rows (blinded: NO detector verdict column)
rows = []
for _, g in gold.iterrows():
    arts = ", ".join(g["config_artifacts"]) if len(g["config_artifacts"]) else ""
    trailers = "; ".join(f'{h["signal"]} @ {h["sha"]}' for h in g["commit_trailer_hits"]) \
               if len(g["commit_trailer_hits"]) else ""
    excerpt = readme_excerpt(g["full_name"], g.get("default_branch") or "main")
    rows.append({
        "repo_id": g["repo_id"],
        "repo_url": f'https://github.com/{g["full_name"]}',
        "language": g["language"],
        "config_artifacts_found": arts,
        "commit_trailer_snippets": trailers,
        "readme_excerpt": excerpt,
        "label": "",          # coder fills: ADOPTED / NOT_ADOPTED / UNSURE
        "reason": "",         # coder fills: one line
    })
evidence = pd.DataFrame(rows)

# identical sheets for both coders
for coder in ("A","B"):
    evidence.to_csv(DIRS['gold'] / f'gold_labeling_coder{coder}.csv',
                    index=False, quoting=csv.QUOTE_ALL)

# calibration batch: first 25 (shared, code these first)
evidence.head(25).to_csv(DIRS['gold'] / 'calibration_batch.csv',
                         index=False, quoting=csv.QUOTE_ALL)

CODEBOOK = textwrap.dedent("""\
    G1 ADOPTION-DETECTION CODEBOOK  (both coders use this identically)
    ==================================================================
    CONSTRUCT: Did THIS repository's own developers use an AI CODING ASSISTANT
    (e.g., GitHub Copilot, Cursor, aider, Claude Code) in producing the code in
    this repo? Judge the project's OWN development, not what the project is about.

    LABEL = ADOPTED  when there is DIRECT evidence of the project's developers
      using an AI coding assistant, e.g.:
        * a genuine commit co-authored by an AI tool (Co-authored-by: Copilot, aider);
        * a project-level AI-tool config that governs THIS codebase
          (CLAUDE.md, .cursor/, .cursorrules, .github/copilot-instructions.md, .aider*);
        * an explicit statement by maintainers that they use such a tool to build this repo.

    LABEL = NOT_ADOPTED  when:
        * the only mention is in documentation/lists/tutorials ABOUT AI tools
          (e.g., an awesome-list entry, a how-to) -- the repo is about AI tools,
          it does not use one to build itself;
        * the artifact is a dependency that merely SUPPORTS AI, not evidence of use;
        * a config file is an empty template / boilerplate with no real use;
        * no relevant evidence at all.

    LABEL = UNSURE  only when evidence is genuinely ambiguous after inspecting
      the repo_url. Add a one-line reason. (UNSURE rows are adjudicated, not dropped.)

    PROCEDURE:
      1) CALIBRATION first: both coders label calibration_batch.csv (25 rows)
         INDEPENDENTLY, then compare and refine shared understanding of edge cases.
      2) Then each coder labels their FULL gold_labeling_coderX.csv INDEPENDENTLY
         (do not look at the other coder's file or discuss while labeling).
      3) Open the repo_url when the in-sheet evidence is insufficient.
      4) Return the completed CSV unchanged in structure (keep repo_id intact).
    """)
(DIRS['gold'] / 'CODEBOOK.txt').write_text(CODEBOOK)

print("Wrote blinded labeling sheets + codebook + calibration batch to:", DIRS['gold'])
for p in ['gold_labeling_coderA.csv','gold_labeling_coderB.csv','calibration_batch.csv','CODEBOOK.txt']:
    print("  ", p)
print("\nNEXT (human step): both coders read CODEBOOK.txt, label calibration_batch.csv,")
print("reconcile, then independently label their coderA / coderB CSVs and return them.")


---
## Cell 7 — Read back labels, compute κ, adjudicate, and run the G1 gate

**Run this only after both coders return their CSVs.** Place the completed files at
`gold/gold_labeling_coderA.csv` and `gold/gold_labeling_coderB.csv` on Drive.

This cell: validates the two files align by `repo_id`; computes **Cohen's κ** on the pre-reconciliation labels (inter-coder reliability for the paper); surfaces disagreements and `UNSURE` rows for **adjudication**; then, once an adjudicated gold label exists, computes **precision / recall / F1 with Wilson 95% CIs** for the detector — the **G1 gate** (precision ≥ 0.90 on high-confidence signals → proceed).


In [ ]:

from sklearn.metrics import cohen_kappa_score, confusion_matrix
from statsmodels.stats.proportion import proportion_confint

A = pd.read_csv(DIRS['gold'] / 'gold_labeling_coderA.csv', dtype=str).set_index('repo_id')
B = pd.read_csv(DIRS['gold'] / 'gold_labeling_coderB.csv', dtype=str).set_index('repo_id')
assert set(A.index) == set(B.index), "Coder files do not share the same repo_id set!"
B = B.reindex(A.index)

def norm(x): return (x or "").strip().upper()
a_lab = A['label'].map(norm); b_lab = B['label'].map(norm)
valid = {"ADOPTED","NOT_ADOPTED","UNSURE"}
assert a_lab.isin(valid).all() and b_lab.isin(valid).all(), \
    "Labels must be ADOPTED / NOT_ADOPTED / UNSURE."

# ---- Cohen's kappa (exclude UNSURE from reliability, report count separately) ----
both_known = (a_lab!="UNSURE") & (b_lab!="UNSURE")
kappa = cohen_kappa_score(a_lab[both_known], b_lab[both_known])
n_unsure = int((~both_known).sum())
agree = float((a_lab[both_known]==b_lab[both_known]).mean())
print(f"Cohen's kappa (both decided, n={both_known.sum()}): {kappa:.3f}")
print(f"Raw agreement: {agree*100:.1f}%   | rows with an UNSURE: {n_unsure}")
interp = ("almost perfect" if kappa>0.8 else "substantial" if kappa>0.6 else
          "moderate" if kappa>0.4 else "fair/poor -> refine codebook & recode")
print("Interpretation:", interp)

# ---- adjudication queue: disagreements + any UNSURE ----
disagree = a_lab.ne(b_lab) | ~both_known
adj_path = DIRS['gold'] / 'adjudication.csv'
if not adj_path.exists():
    q = pd.DataFrame({"repo_id": a_lab.index, "coderA": a_lab.values, "coderB": b_lab.values,
                      "repo_url": A['repo_url'].values, "adjudicated_label": ""})
    q[disagree.values].to_csv(adj_path, index=False)
    print(f"\n{int(disagree.sum())} rows need adjudication -> {adj_path}")
    print("Fill 'adjudicated_label' (ADOPTED/NOT_ADOPTED) for those rows, then re-run this cell.")
else:
    # ---- build final gold labels: agreements stand; adjudicated fill the rest ----
    adj = pd.read_csv(adj_path, dtype=str).set_index('repo_id')
    final = pd.Series(index=a_lab.index, dtype=object)
    final[~disagree] = a_lab[~disagree]
    for rid in adj.index:
        v = norm(adj.loc[rid,'adjudicated_label'])
        if v in {"ADOPTED","NOT_ADOPTED"}: final[rid] = v
    if final.isna().any() or (final=="UNSURE").any():
        miss = final[final.isna() | (final=="UNSURE")].index.tolist()
        print(f"Still need adjudicated labels for {len(miss)} rows: {miss[:10]} ...")
    else:
        gold = pd.read_parquet(DIRS['gold'] / 'gold_set.parquet').set_index('repo_id')
        y_true = (final.reindex(gold.index)=="ADOPTED").astype(int)
        y_pred = gold['high_conf_signal'].astype(int)   # detector verdict
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        def wilson(k,n):
            if n==0: return (float('nan'),float('nan'))
            lo,hi = proportion_confint(k,n,alpha=0.05,method='wilson'); return (lo,hi)
        prec = tp/(tp+fp) if (tp+fp) else float('nan')
        rec  = tp/(tp+fn) if (tp+fn) else float('nan')
        f1   = 2*prec*rec/(prec+rec) if prec and rec and (prec+rec) else float('nan')
        p_lo,p_hi = wilson(tp, tp+fp); r_lo,r_hi = wilson(tp, tp+fn)

        gate = {
            "cohens_kappa": round(kappa,3), "raw_agreement": round(agree,3),
            "confusion": {"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn)},
            "precision": round(prec,3), "precision_95CI": [round(p_lo,3),round(p_hi,3)],
            "recall": round(rec,3), "recall_95CI": [round(r_lo,3),round(r_hi,3)],
            "f1": round(f1,3),
            "GATE_CRITERION": "precision >= 0.90 (use 95% CI lower bound) on high-conf signals",
            "GATE_PASSED": bool(p_lo >= 0.90),
        }
        (DIRS['results'] / 'G1_gate.json').write_text(json.dumps(gate, indent=2))
        print("\n================ G1 VALIDATION GATE ================")
        print(json.dumps(gate, indent=2))
        print("\nDECISION:",
              "PASS -> proceed to Notebook 3 (panel construction + DiD)."
              if gate["GATE_PASSED"] else
              "FAIL -> pivot to the methods-contribution branch (per pre-registration); "
              "do an error analysis of false positives before scaling.")


---
## Cell 8 — Manifest & handoff to Notebook 3


In [ ]:

manifest = {
    "notebook": "02_frame_and_G1_validation",
    "produced": {
        "frame/frame_final.parquet": (DIRS['frame']/'frame_final.parquet').exists(),
        "frame/attrition.json": (DIRS['frame']/'attrition.json').exists(),
        "gold/gold_set.parquet": (DIRS['gold']/'gold_set.parquet').exists(),
        "gold/CODEBOOK.txt": (DIRS['gold']/'CODEBOOK.txt').exists(),
        "gold/gold_labeling_coderA.csv": (DIRS['gold']/'gold_labeling_coderA.csv').exists(),
        "gold/gold_labeling_coderB.csv": (DIRS['gold']/'gold_labeling_coderB.csv').exists(),
        "results/G1_gate.json": (DIRS['results']/'G1_gate.json').exists(),
    },
    "timestamp": datetime.datetime.utcnow().isoformat()+"Z",
}
(PROJECT_ROOT / 'manifest_nb02.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("\nIf G1 PASSED: Notebook 3 builds the repo x month panels (activity from git history "
      "= full window; CI-minutes scoped to the retrievable ~14-month window), then matched DiD.")
